In [1]:
import pandas as pd
import numpy as np
import joblib

# Cargamos el modelo de PD ya entrenado
modelo_pd = joblib.load('../modelo_pd_xgb.pkl')

# Cargamos el dataset completo de nuevo (mismas columnas que usamos para PD)
df = pd.read_csv('../data/accepted_2007_to_2018Q4.csv.gz', low_memory=False)

cols = ['funded_amnt', 'term', 'int_rate', 'grade', 'emp_length',
        'home_ownership', 'annual_inc', 'purpose', 'dti', 'loan_status']
df = df[cols]

# Nos quedamos con préstamos de resultado definitivo (mismo criterio que en PD)
df = df[df['loan_status'].isin(['Fully Paid', 'Charged Off'])].copy()

print(f"Total de préstamos: {len(df):,}")

Total de préstamos: 1,345,310


In [2]:
# Preparamos las features, mismo proceso que en el modelo de PD
features = ['funded_amnt', 'term', 'int_rate', 'grade', 'emp_length',
            'home_ownership', 'annual_inc', 'purpose', 'dti']

X_full = df[features].copy()
X_full = X_full.rename(columns={'funded_amnt': 'loan_amnt'})  # el modelo espera este nombre

X_full['term'] = X_full['term'].str.extract(r'(\d+)').astype(float)

emp_map = {'< 1 year': 0, '1 year': 1, '2 years': 2, '3 years': 3, '4 years': 4,
           '5 years': 5, '6 years': 6, '7 years': 7, '8 years': 8, '9 years': 9, '10+ years': 10}
X_full['emp_length'] = X_full['emp_length'].map(emp_map)

grade_map = {'A': 1, 'B': 2, 'C': 3, 'D': 4, 'E': 5, 'F': 6, 'G': 7}
X_full['grade'] = X_full['grade'].map(grade_map)

X_full = pd.get_dummies(X_full, columns=['home_ownership', 'purpose'], drop_first=True)

X_full['emp_length'] = X_full['emp_length'].fillna(X_full['emp_length'].median())
X_full['dti'] = X_full['dti'].fillna(X_full['dti'].median())

# Predecimos la probabilidad de default (PD) para cada préstamo
df['PD'] = modelo_pd.predict_proba(X_full)[:, 1]

df[['funded_amnt', 'grade', 'PD']].head()

,funded_amnt,grade,PD
0,3600.0,C,0.115503
1,24700.0,C,0.255539
2,20000.0,B,0.171966
4,10400.0,F,0.371160
5,11950.0,C,0.224306


In [3]:
# Diccionario de LGD por grado (del Paso 3)
lgd_dict = {'A': 0.403, 'B': 0.395, 'C': 0.407, 'D': 0.408, 'E': 0.401, 'F': 0.404, 'G': 0.435}

# Asignamos la LGD correspondiente según el grado de cada préstamo
df['LGD'] = df['grade'].map(lgd_dict)

# EAD = Exposure at Default. Usamos el monto del préstamo como aproximación
df['EAD'] = df['funded_amnt']

# Expected Loss por préstamo
df['expected_loss'] = df['PD'] * df['LGD'] * df['EAD']

df[['funded_amnt', 'grade', 'PD', 'LGD', 'EAD', 'expected_loss']].head()

,funded_amnt,grade,PD,LGD,EAD,expected_loss
0,3600.0,C,0.115503,0.407,3600.0,169.234797
1,24700.0,C,0.255539,0.407,24700.0,2568.904119
2,20000.0,B,0.171966,0.395,20000.0,1358.534589
4,10400.0,F,0.371160,0.404,10400.0,1559.466359
5,11950.0,C,0.224306,0.407,11950.0,1090.947337


In [4]:
el_total = df['expected_loss'].sum()
ead_total = df['EAD'].sum()
el_ratio = el_total / ead_total * 100

print(f"Expected Loss total del portafolio: ${el_total:,.2f}")
print(f"Exposición total (EAD) del portafolio: ${ead_total:,.2f}")
print(f"Expected Loss ratio: {el_ratio:.2f}%")

Expected Loss total del portafolio: $1,687,783,507.57
Exposición total (EAD) del portafolio: $19,388,011,250.00
Expected Loss ratio: 8.71%
